# Feature Engineering

## Objetivo

Construir las estructuras necesarias para entrenar sistemas de recomendación basados en interacciones entre usuarios y productos.

En esta etapa se generarán las variables y matrices que permitirán modelar las preferencias de los usuarios.

In [1]:
# Importar librerías necesarias

import pandas as pd
import numpy as np

In [2]:
# Cargar dataset procesado con columnas necesarias

cols = [
    "user_id",
    "product_id",
    "reordered"
]

interactions = pd.read_parquet(
    "../data/processed/interactions_filtered.parquet",
    columns=cols
)

interactions.head()

,user_id,product_id,reordered
0,202279,33120,1
1,202279,28985,1
2,202279,9327,0
3,202279,45918,1
4,202279,30035,0


In [3]:
# Revisar dimensiones

interactions.shape

(32300077, 3)

In [4]:
# Revisar dimensiones

print("Interacciones:", len(interactions))
print("Usuarios:", interactions["user_id"].nunique())
print("Productos:", interactions["product_id"].nunique())

Interacciones: 32300077
Usuarios: 206208
Productos: 35922


In [5]:
# Calcular cantidad de compras por usuario

user_features = (
    interactions
    .groupby("user_id")
    .agg(
        total_interactions=("product_id", "count"),
        unique_products=("product_id", "nunique"),
        reorder_rate=("reordered", "mean")
    )
    .reset_index()
)

user_features.head()

,user_id,total_interactions,unique_products,reorder_rate
0,1,59,18,0.694915
1,2,195,102,0.476923
2,3,88,33,0.625000
3,4,17,16,0.058824
4,5,37,23,0.378378


In [6]:
# Estadísticas descriptivas de usuarios

user_features.describe()

,user_id,total_interactions,unique_products,reorder_rate
count,206208.000000,206208.000000,206208.000000,206208.000000
mean,103105.178276,156.638331,64.041332,0.433091
std,59527.644457,203.739244,56.132415,0.212253
min,1.000000,2.000000,1.000000,0.000000
25%,51552.750000,39.000000,25.000000,0.269231
50%,103105.500000,83.000000,47.000000,0.430303
75%,154657.250000,187.000000,85.000000,0.596964
max,206209.000000,3725.000000,724.000000,0.989529


In [7]:
# Construir métricas de producto

product_features = (
    interactions
    .groupby("product_id")
    .agg(
        total_purchases=("user_id", "count"),
        unique_users=("user_id", "nunique"),
        reorder_rate=("reordered", "mean")
    )
    .reset_index()
)

product_features.head()

,product_id,total_purchases,unique_users,reorder_rate
0,1,1852,716,0.613391
1,2,90,78,0.133333
2,3,277,74,0.732852
3,4,329,182,0.446809
4,7,30,18,0.400000


In [8]:
# Estadísticas descriptivas de productos

product_features.describe()

,product_id,total_purchases,unique_users,reorder_rate
count,35922.000000,35922.000000,35922.000000,35922.000000
mean,24848.086075,899.172568,367.625271,0.429594
std,14337.507434,5615.951732,1527.388972,0.178498
min,1.000000,20.000000,2.000000,0.000000
25%,12519.500000,48.000000,29.000000,0.300000
50%,24807.500000,126.000000,71.000000,0.441757
75%,37248.750000,443.000000,226.000000,0.564588
max,49688.000000,472565.000000,73956.000000,0.941176


In [9]:
# Usuarios con al menos 50 interacciones

active_users = user_features[
    user_features["total_interactions"] >= 50
]["user_id"]

len(active_users)

138882

In [10]:
# Dataset reducido para modelado inicial

model_data = (
    interactions
    .sample(
        n=3_000_000,
        random_state=42
    )
    .reset_index(drop=True)
)

model_data.shape

(3000000, 3)

## Selección de muestra para modelado

Debido al gran volumen del dataset original, se generó una muestra reproducible de 3.000.000 de interacciones para las primeras etapas de modelado.

Esta decisión permite entrenar y comparar modelos de forma eficiente, evitando problemas de memoria durante el desarrollo.

La muestra conserva la estructura principal del problema usuario-producto y permite construir una primera versión funcional del sistema de recomendación.

In [11]:
# Guardar dataset para modelado

model_data.to_parquet(
    "../data/processed/model_data.parquet",
    index=False
)

In [12]:
# Verificar que el archivo fue creado correctamente

import os

print(
    os.path.exists(
        "../data/processed/model_data.parquet"
    )
)

True


In [13]:
# Verificar que el archivo fue creado correctamente

import os

os.path.exists(
    "../data/processed/model_data.parquet"
)

True

# Conclusiones de Feature Engineering

## Features generadas

Se construyeron métricas descriptivas para usuarios y productos:

### Usuarios

* Total de interacciones.
* Cantidad de productos únicos consumidos.
* Tasa de recompra.

### Productos

* Cantidad total de compras.
* Cantidad de usuarios únicos.
* Tasa de recompra.

## Selección de muestra para modelado

Debido al gran volumen del dataset original, se generó una muestra reproducible de 3.000.000 de interacciones para las primeras etapas de modelado.

Esta decisión permite entrenar y comparar modelos de forma eficiente, evitando problemas de memoria durante el desarrollo.

La muestra conserva la estructura principal del problema usuario-producto y permite construir una primera versión funcional del sistema de recomendación.

## Dataset para modelado

Se generó un dataset de interacciones usuario-producto con las columnas necesarias para entrenar modelos de recomendación:

* user_id
* product_id
* reordered

El dataset resultante será utilizado como base para la construcción y evaluación de los distintos modelos de recomendación propuestos.


## Implicancias para el modelado

La selección de una muestra representativa permite reducir los requerimientos computacionales y facilitar la experimentación con distintos algoritmos de recomendación.

El dataset resultante será utilizado para entrenar y comparar distintos enfoques de recomendación.

## Modelos a evaluar

- Baseline basado en popularidad.
- Collaborative Filtering basado en usuarios.
- Exploración de técnicas de Market Basket Analysis para recomendaciones complementarias y estrategias de Cross Selling.